# MAIAC HDF locator & inventory (HDF4‑EOS)

Notebook này giúp bạn:

1) **Tìm file MAIAC** trong repo (quét `*.hdf`, `*.hdf4`, `*.he5`) – mặc định trỏ vào `crawler/maiac_data`.  
2) **Auto-detect** file là **HDF4** hay **HDF5** bằng magic bytes, để biết nên dùng `pyhdf/GDAL` hay `h5py`.  
3) **Parse filename** (product / AYYYYDDD / tile h??v?? / collection).  
4) (Optional) Xuất inventory thành Parquet: `clean/maiac_file_inventory.parquet` (ở *repo root*).

> Chạy từ trên xuống. Nếu repo có nhiều file lớn, cell scan có thể mất vài giây.

In [4]:
from __future__ import annotations

import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal, Optional

import pandas as pd

# --- Paths ---
# Notebook runtime không có __file__, nên fallback theo cwd.
# Mặc định: nếu đang chạy trong `notebooks/` thì repo root = parent.
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd.parent if _cwd.name.lower() == "notebooks" else _cwd
MAIAC_DIR = REPO_ROOT / "crawler" / "maiac_data"

# Scan mode:
# - "maiac_dir": nhanh, đúng nơi đang chứa file MAIAC trong repo
# - "repo": quét toàn repo nếu bạn muốn tìm thêm file HDF ở nơi khác
SCAN_MODE: Literal["maiac_dir", "repo"] = "maiac_dir"

start_dir = MAIAC_DIR if SCAN_MODE == "maiac_dir" else REPO_ROOT

print("CWD:", _cwd)
print("Repo root:", REPO_ROOT)
print("Start dir:", start_dir)
print("Exists:", start_dir.exists())

"""1) Locate MAIAC HDF/HDF4-EOS data in repo (glob search + summary)"""

EXTS = [".hdf", ".hdf4", ".he5"]

files: list[Path] = []
for ext in EXTS:
    files.extend(start_dir.rglob(f"*{ext}"))

files = sorted(set(files))

print("Total candidate files:", len(files))
print("First 20:")
for p in files[:20]:
    try:
        size_mb = p.stat().st_size / (1024 * 1024)
    except Exception:
        size_mb = None
    print(f"- {p.as_posix()}  ({'' if size_mb is None else f'{size_mb:.2f} MB'})")

# Summary theo folder
rows = []
for p in files:
    try:
        rows.append({
            "dir": str(p.parent),
            "file": p.name,
            "size_bytes": p.stat().st_size,
            "ext": p.suffix.lower(),
        })
    except Exception:
        rows.append({
            "dir": str(p.parent),
            "file": p.name,
            "size_bytes": None,
            "ext": p.suffix.lower(),
        })

df_all = pd.DataFrame(rows)
if not df_all.empty:
    display(df_all.head(10))

    print("\nTop directories (by file count):")
    display(
        df_all.groupby("dir", as_index=False)
        .agg(file_count=("file", "count"), total_size_mb=("size_bytes", lambda s: float(s.fillna(0).sum()) / (1024 * 1024)))
        .sort_values(["file_count", "total_size_mb"], ascending=False)
        .head(20)
    )
else:
    print("No .hdf/.hdf4/.he5 files found.")

"""2) Auto-detect MAIAC file type (HDF4 vs HDF5) and suggest reader"""

HDF4_MAGIC = b"\x0e\x03\x13\x01"  # HDF4 signature
HDF5_MAGIC = b"\x89HDF\r\n\x1a\n"  # HDF5 signature

FileFormat = Literal["HDF4", "HDF5", "unknown"]


def detect_hdf_format(path: Path) -> FileFormat:
    try:
        with path.open("rb") as f:
            head = f.read(16)
        if head.startswith(HDF4_MAGIC):
            return "HDF4"
        if head.startswith(HDF5_MAGIC):
            return "HDF5"
        return "unknown"
    except Exception:
        return "unknown"


sample_n = min(10, len(files))
print(f"\nDetecting format for a sample of {sample_n} files...")
for p in files[:sample_n]:
    fmt = detect_hdf_format(p)
    print(f"- {fmt:7s}  {p.as_posix()}")

print("\nGuidance:")
print("- If format=HDF4: use pyhdf (HDF4-EOS) or GDAL.")
print("- If format=HDF5 / HE5: use h5py / h5netcdf.")

"""3) Print candidate directories + sample filenames (tile/date parsing check)"""

# Pattern: MCD19A2.AYYYYDDD.hXXvYY.061.YYYYDDDHHMMSS.hdf
MAIAC_RE = re.compile(
    r"^(?P<product>[A-Z0-9]+)\.A(?P<year>\d{4})(?P<doy>\d{3})\.h(?P<h>\d{2})v(?P<v>\d{2})\.(?P<collection>\d{3})\.(?P<proc_ts>\d+)\.(?P<ext>hdf|hdf4)$",
    re.IGNORECASE,
)


@dataclass
class MaiacMeta:
    product: str
    date: Optional[str]
    year: Optional[int]
    doy: Optional[int]
    h_tile: Optional[int]
    v_tile: Optional[int]
    tile_id: Optional[str]
    collection: Optional[str]
    processing_ts: Optional[str]


def parse_maiac_filename(filename: str) -> Optional[MaiacMeta]:
    m = MAIAC_RE.match(filename)
    if not m:
        return None

    year = int(m.group("year"))
    doy = int(m.group("doy"))
    h_tile = int(m.group("h"))
    v_tile = int(m.group("v"))

    try:
        # date string YYYY-MM-DD from year + DOY
        date = pd.to_datetime(f"{year}-{doy:03d}", format="%Y-%j").date().isoformat()
    except Exception:
        date = None

    return MaiacMeta(
        product=m.group("product"),
        date=date,
        year=year,
        doy=doy,
        h_tile=h_tile,
        v_tile=v_tile,
        tile_id=f"h{h_tile:02d}v{v_tile:02d}",
        collection=m.group("collection"),
        processing_ts=m.group("proc_ts"),
    )


# Filter only filenames matching MAIAC style
maiac_candidates = []
fails = []
for p in files:
    meta = parse_maiac_filename(p.name)
    if meta is None:
        fails.append(p)
        continue
    maiac_candidates.append((p, meta))

print("\nMAIAC-style candidates:", len(maiac_candidates))
print("Non-matching HDF files:", len(fails))

# Show a sample
for p, meta in maiac_candidates[:10]:
    print("-", p.as_posix(), "->", meta)

# Quick tile distribution check
if maiac_candidates:
    df_meta = pd.DataFrame([
        {
            "path": str(p),
            "dir": str(p.parent),
            "filename": p.name,
            **asdict(meta),
        }
        for p, meta in maiac_candidates
    ])

    print("\nTile distribution (count):")
    display(df_meta.groupby("tile_id", as_index=False).agg(n=("filename", "count")).sort_values("n", ascending=False))

    print("\nMin/max date:")
    display(df_meta[["date"]].dropna().agg(["min", "max"]))

"""4) (Optional) Build an index parquet of all MAIAC files for the pipeline"""

out_dir = REPO_ROOT / "clean"
out_dir.mkdir(parents=True, exist_ok=True)

inventory_rows = []
for p in files:
    fmt = detect_hdf_format(p)
    meta = parse_maiac_filename(p.name)
    inventory_rows.append({
        "path": str(p),
        "dir": str(p.parent),
        "filename": p.name,
        "ext": p.suffix.lower(),
        "size_bytes": p.stat().st_size if p.exists() else None,
        "format": fmt,
        # parsed
        "product": None if meta is None else meta.product,
        "date": None if meta is None else meta.date,
        "year": None if meta is None else meta.year,
        "doy": None if meta is None else meta.doy,
        "tile_id": None if meta is None else meta.tile_id,
        "h_tile": None if meta is None else meta.h_tile,
        "v_tile": None if meta is None else meta.v_tile,
        "collection": None if meta is None else meta.collection,
        "processing_ts": None if meta is None else meta.processing_ts,
    })

df_inv = pd.DataFrame(inventory_rows)

inv_path = out_dir / "maiac_file_inventory.parquet"
df_inv.to_parquet(inv_path, index=False)
print("Saved inventory:", inv_path.as_posix())

# Show a filtered view of likely MAIAC HDF4 files
if not df_inv.empty:
    display(
        df_inv[(df_inv["format"] == "HDF4") & (df_inv["product"].notna())]
        .sort_values(["date", "tile_id"], ascending=True)
        .head(50)
    )

CWD: E:\Big-data-in-Finance\notebooks
Repo root: E:\Big-data-in-Finance
Start dir: E:\Big-data-in-Finance\crawler\maiac_data
Exists: True
Total candidate files: 21
First 20:
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h27v06.061.2026090153844.hdf  (9.86 MB)
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h27v07.061.2026090151936.hdf  (12.68 MB)
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h28v06.061.2026090151609.hdf  (6.29 MB)
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h28v07.061.2026090153124.hdf  (7.50 MB)
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h28v08.061.2026090150826.hdf  (4.58 MB)
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h29v07.061.2026090153226.hdf  (7.57 MB)
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h29v08.061.2026090150630.hdf  (4.09 MB)
- E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026088.h27v06.061.2026090160101.hdf  (7.90 MB)
- E:/Big-data-in-

,dir,file,size_bytes,ext
0,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h27v06.061.2026090153844.hdf,10342704,.hdf
1,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h27v07.061.2026090151936.hdf,13300605,.hdf
2,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h28v06.061.2026090151609.hdf,6598609,.hdf
3,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h28v07.061.2026090153124.hdf,7865287,.hdf
4,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h28v08.061.2026090150826.hdf,4797972,.hdf
5,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h29v07.061.2026090153226.hdf,7941774,.hdf
6,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h29v08.061.2026090150630.hdf,4291510,.hdf
7,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026088.h27v06.061.2026090160101.hdf,8286183,.hdf
8,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026088.h27v07.061.2026090154343.hdf,9505996,.hdf
9,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026088.h28v06.061.2026090153401.hdf,7456703,.hdf



Top directories (by file count):


,dir,file_count,total_size_mb
0,E:\Big-data-in-Finance\crawler\maiac_data,21,152.840446



Detecting format for a sample of 10 files...
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h27v06.061.2026090153844.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h27v07.061.2026090151936.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h28v06.061.2026090151609.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h28v07.061.2026090153124.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h28v08.061.2026090150826.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h29v07.061.2026090153226.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026087.h29v08.061.2026090150630.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026088.h27v06.061.2026090160101.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2026088.h27v07.061.2026090154343.hdf
- HDF4     E:/Big-data-in-Finance/crawler/maiac_data/MCD19A2.A2

,tile_id,n
0,h27v06,3
1,h27v07,3
2,h28v06,3
3,h28v07,3
4,h28v08,3
5,h29v07,3
6,h29v08,3



Min/max date:


,date
min,2026-03-28
max,2026-03-30


Saved inventory: E:/Big-data-in-Finance/clean/maiac_file_inventory.parquet


,path,dir,filename,ext,size_bytes,format,product,date,year,doy,tile_id,h_tile,v_tile,collection,processing_ts
0,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h27v06.061.2026090153844.hdf,.hdf,10342704,HDF4,MCD19A2,2026-03-28,2026,87,h27v06,27,6,061,2026090153844
1,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h27v07.061.2026090151936.hdf,.hdf,13300605,HDF4,MCD19A2,2026-03-28,2026,87,h27v07,27,7,061,2026090151936
2,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h28v06.061.2026090151609.hdf,.hdf,6598609,HDF4,MCD19A2,2026-03-28,2026,87,h28v06,28,6,061,2026090151609
3,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h28v07.061.2026090153124.hdf,.hdf,7865287,HDF4,MCD19A2,2026-03-28,2026,87,h28v07,28,7,061,2026090153124
4,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h28v08.061.2026090150826.hdf,.hdf,4797972,HDF4,MCD19A2,2026-03-28,2026,87,h28v08,28,8,061,2026090150826
5,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h29v07.061.2026090153226.hdf,.hdf,7941774,HDF4,MCD19A2,2026-03-28,2026,87,h29v07,29,7,061,2026090153226
6,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026087.h29v08.061.2026090150630.hdf,.hdf,4291510,HDF4,MCD19A2,2026-03-28,2026,87,h29v08,29,8,061,2026090150630
7,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026088.h27v06.061.2026090160101.hdf,.hdf,8286183,HDF4,MCD19A2,2026-03-29,2026,88,h27v06,27,6,061,2026090160101
8,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026088.h27v07.061.2026090154343.hdf,.hdf,9505996,HDF4,MCD19A2,2026-03-29,2026,88,h27v07,27,7,061,2026090154343
9,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...,E:\Big-data-in-Finance\crawler\maiac_data,MCD19A2.A2026088.h28v06.061.2026090153401.hdf,.hdf,7456703,HDF4,MCD19A2,2026-03-29,2026,88,h28v06,28,6,061,2026090153401


In [5]:
# Preview processed MAIAC inventory (the output we just wrote)
import pandas as pd

inv_path = REPO_ROOT / "clean" / "maiac_file_inventory.parquet"
print("Inventory path:", inv_path)
print("Exists:", inv_path.exists())

if inv_path.exists():
    df = pd.read_parquet(inv_path)
    print("Shape:", df.shape)

    # Show a small sample of parsed MAIAC rows
    display(
        df[(df["product"].notna()) & (df["format"] == "HDF4")]
        .sort_values(["date", "tile_id", "filename"], ascending=True)
        [["date", "tile_id", "product", "collection", "format", "size_bytes", "filename", "path"]]
        .head(15)
    )

    print("\nDate range:")
    display(df[["date"]].dropna().agg(["min", "max"]))

    print("\nTile distribution:")
    display(
        df[df["tile_id"].notna()]
        .groupby("tile_id", as_index=False)
        .agg(n_files=("filename", "count"), total_mb=("size_bytes", lambda s: float(s.fillna(0).sum()) / (1024 * 1024)))
        .sort_values(["n_files", "total_mb"], ascending=False)
    )
else:
    print("Inventory not found yet. Run the scan cell above first.")


Inventory path: E:\Big-data-in-Finance\clean\maiac_file_inventory.parquet
Exists: True
Shape: (21, 15)


,date,tile_id,product,collection,format,size_bytes,filename,path
0,2026-03-28,h27v06,MCD19A2,061,HDF4,10342704,MCD19A2.A2026087.h27v06.061.2026090153844.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
1,2026-03-28,h27v07,MCD19A2,061,HDF4,13300605,MCD19A2.A2026087.h27v07.061.2026090151936.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
2,2026-03-28,h28v06,MCD19A2,061,HDF4,6598609,MCD19A2.A2026087.h28v06.061.2026090151609.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
3,2026-03-28,h28v07,MCD19A2,061,HDF4,7865287,MCD19A2.A2026087.h28v07.061.2026090153124.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
4,2026-03-28,h28v08,MCD19A2,061,HDF4,4797972,MCD19A2.A2026087.h28v08.061.2026090150826.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
5,2026-03-28,h29v07,MCD19A2,061,HDF4,7941774,MCD19A2.A2026087.h29v07.061.2026090153226.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
6,2026-03-28,h29v08,MCD19A2,061,HDF4,4291510,MCD19A2.A2026087.h29v08.061.2026090150630.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
7,2026-03-29,h27v06,MCD19A2,061,HDF4,8286183,MCD19A2.A2026088.h27v06.061.2026090160101.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
8,2026-03-29,h27v07,MCD19A2,061,HDF4,9505996,MCD19A2.A2026088.h27v07.061.2026090154343.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...
9,2026-03-29,h28v06,MCD19A2,061,HDF4,7456703,MCD19A2.A2026088.h28v06.061.2026090153401.hdf,E:\Big-data-in-Finance\crawler\maiac_data\MCD1...



Date range:


,date
min,2026-03-28
max,2026-03-30



Tile distribution:


,tile_id,n_files,total_mb
1,h27v07,3,30.859076
0,h27v06,3,27.657394
3,h28v07,3,25.862218
5,h29v07,3,20.520028
2,h28v06,3,19.443661
6,h29v08,3,16.393401
4,h28v08,3,12.104669
